In [ ]:
"""
Sample paths of the d-dimensional Elephant Random Walk (ERW) in d = 3,
for two values of the memory parameter p.

For Section 7.2 (multi-dimensional ERW) of the thesis.

Style matches Figure 2.1: ITALIC_RED line on white, minimal axes,
Palatino / STIX typography.

ERW convention (multi-dimensional generalization, Bercu & Laulin 2019):
    - eta_1 is uniform on the 2d unit vectors {+e_1, -e_1, ..., +e_d, -e_d}.
    - For n >= 1: choose K uniform on {1, ..., n}. Then:
        * with probability p, set eta_{n+1} = eta_K (repeat);
        * with probability (1 - p), set eta_{n+1} to one of the
          2d - 1 other unit vectors, each with equal probability (1 - p)/(2d - 1).
    - X_n = sum_{k=1}^{n} eta_k.
"""

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.colors import to_rgb
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers 3d projection)
from mpl_toolkits.mplot3d.art3d import Line3DCollection

# ---------------------------------------------------------------------------
# Thesis style
# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# Thesis style
# ---------------------------------------------------------------------------
# Original palette:
#   \definecolor{tropicalrainforest}{rgb}{0.0, 0.46, 0.37}
#   \definecolor{ItalicRed}{RGB}{177,56,69}
# We use slightly softened versions for the path so that 6,000 overlapping
# segments don't read as a heavy block of saturated colour.
TROPICAL_RAINFOREST = "#1B7766"   # softened from rgb(0.00, 0.46, 0.37)
ITALIC_RED          = "#C25966"   # softened from RGB(177, 56, 69)

TITLE_GREY  = "#888888"
SLATE_STEEL = "#4A72A4"           # \definecolor{SlateSteel}{RGB}{74, 114, 164}

rcParams["font.family"]       = "serif"
rcParams["font.serif"]        = ["Palatino", "Palatino Linotype",
                                  "URW Palladio L", "Book Antiqua",
                                  "STIX Two Text", "STIXGeneral",
                                  "DejaVu Serif"]
rcParams["mathtext.fontset"]  = "stix"
rcParams["axes.linewidth"]    = 0.6
rcParams["xtick.major.width"] = 0.6
rcParams["ytick.major.width"] = 0.6
rcParams["font.size"]         = 9


# ---------------------------------------------------------------------------
# d-dimensional ERW simulator
# ---------------------------------------------------------------------------
def simulate_erw(n_steps, p, d=3, seed=None):
    """Return an (n_steps + 1, d) array of positions X_0, X_1, ..., X_n."""
    rng = np.random.default_rng(seed)

    # Enumerate the 2d unit vectors as rows of a (2d, d) integer matrix:
    # rows 0..d-1 are +e_1..+e_d, rows d..2d-1 are -e_1..-e_d.
    basis = np.zeros((2 * d, d), dtype=np.int64)
    for j in range(d):
        basis[j, j]      =  1
        basis[d + j, j]  = -1

    # We store each eta as the index (0..2d-1) of its unit vector in `basis`.
    eta_idx = np.empty(n_steps, dtype=np.int64)

    # Step 1: uniform on all 2d unit vectors.
    eta_idx[0] = rng.integers(0, 2 * d)

    # For step n+1 (n >= 1): K uniform on {0, ..., n-1}, repeat with prob p,
    # else choose one of the 2d - 1 other directions uniformly.
    K_arr   = rng.integers(0, np.arange(1, n_steps), endpoint=False)  # K in {0,...,n-1}
    rep_arr = rng.random(n_steps - 1) < p
    # For the "not repeat" case, draw an offset in {1, ..., 2d - 1} and add
    # mod 2d to the remembered direction. This gives a uniform pick over the
    # remaining 2d - 1 directions.
    offset_arr = rng.integers(1, 2 * d, size=n_steps - 1)

    for n in range(1, n_steps):
        remembered = eta_idx[K_arr[n - 1]]
        if rep_arr[n - 1]:
            eta_idx[n] = remembered
        else:
            eta_idx[n] = (remembered + offset_arr[n - 1]) % (2 * d)

    eta = basis[eta_idx]
    positions = np.zeros((n_steps + 1, d), dtype=np.int64)
    np.cumsum(eta, axis=0, out=positions[1:])
    return positions


# ---------------------------------------------------------------------------
# Plotting
# ---------------------------------------------------------------------------
def plot_path(ax, positions, color, title=None,
              alpha_near=0.95, alpha_far=0.30, gamma=1.0):
    """Draw a single ERW sample path on the given 3D axes.

    The path fades from `alpha_near` near the origin to `alpha_far` at the
    farthest visited point. The fade uses (r / r_max) ** gamma where r is the
    Euclidean distance to the origin.

    No axes, ticks, panes or grid are drawn — only the path and a marker at
    the origin (the starting point of the walk).
    """
    x, y, z = positions[:, 0], positions[:, 1], positions[:, 2]

    # Build segments for Line3DCollection.
    pts  = positions.reshape(-1, 1, 3)            # (N, 1, 3)
    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)   # (N-1, 2, 3)

    # Per-segment distance from origin, taken as the midpoint distance.
    mids = 0.5 * (segs[:, 0, :] + segs[:, 1, :])
    r    = np.linalg.norm(mids, axis=1)
    r_max = r.max() if r.max() > 0 else 1.0
    t = (r / r_max) ** gamma                       # in [0, 1]
    alphas = alpha_near + (alpha_far - alpha_near) * t

    # Build per-segment RGBA colours.
    rgb = np.array(to_rgb(color))
    colors = np.tile(rgb, (segs.shape[0], 1))      # (N-1, 3)
    colors = np.concatenate([colors, alphas[:, None]], axis=1)

    lc = Line3DCollection(segs, colors=colors, linewidths=0.5,
                          capstyle="round", joinstyle="round")
    ax.add_collection3d(lc)

    # Force the axes to actually contain the path (LineCollection does not
    # auto-update the data limits in 3D).
    ax.set_xlim(x.min(), x.max())
    ax.set_ylim(y.min(), y.max())
    ax.set_zlim(z.min(), z.max())

    # Origin marker — clearly visible: SlateSteel-blue dot with a white halo,
    # so the starting point of the walk reads at a glance even against the
    # densest part of the path.
    ax.scatter([0], [0], [0],
               color="white", s=260, zorder=11,
               edgecolors="none",
               depthshade=False)
    ax.scatter([0], [0], [0],
               color=SLATE_STEEL, s=90, zorder=12,
               edgecolors="white", linewidths=1.4,
               depthshade=False)

    # Strip everything: ticks, labels, panes, grid, axis lines.
    ax.set_axis_off()

    # Title is placed by the caller via fig.text() for precise positioning,
    # because with set_axis_off() the axes rectangle is much larger than the
    # visible cube and ax.set_title() floats too far above the plot.


# ---------------------------------------------------------------------------
# Build the figure
# ---------------------------------------------------------------------------
def main():
    n_steps = 6000           # match Figure 2.1
    p_low   = 0.4
    p_high  = 0.7

    # Seeds chosen so both panels are visually clean and non-degenerate.
    pos_low  = simulate_erw(n_steps, p_low,  d=3, seed=11)
    pos_high = simulate_erw(n_steps, p_high, d=3, seed=7)

    fig = plt.figure(figsize=(8.5, 4.4))

    ax1 = fig.add_subplot(1, 2, 1, projection="3d")
    plot_path(ax1, pos_low,  color=TROPICAL_RAINFOREST,
              title=fr"$p = {p_low}$")

    ax2 = fig.add_subplot(1, 2, 2, projection="3d")
    plot_path(ax2, pos_high, color=ITALIC_RED,
              title=fr"$p = {p_high}$")

    # Slightly elevated, lightly rotated view — matches the feel of Figure 2.1's
    # right-hand 3D panel.
    for ax in (ax1, ax2):
        ax.view_init(elev=22, azim=-55)

    fig.subplots_adjust(left=0.0, right=1.0, top=1.0, bottom=0.0,
                        wspace=-0.30)

    # Place the panel titles via fig.text in figure-relative coords, just
    # above where the visible cube sits. With wspace=-0.30 the two panels
    # overlap, so panel centres are ~0.30 and ~0.70.
    fig.text(0.30, 0.84, fr"$p = {p_low}$",
             ha="center", va="bottom",
             fontsize=11, color=TITLE_GREY)
    fig.text(0.70, 0.84, fr"$p = {p_high}$",
             ha="center", va="bottom",
             fontsize=11, color=TITLE_GREY)

    out_pdf = "/home/claude/erw3d/erw3d.pdf"
    out_png = "/home/claude/erw3d/erw3d.png"

    print(f"Wrote {out_pdf}")
    print(f"Wrote {out_png}")

    # A bit of diagnostic output.
    end_low  = pos_low[-1]
    end_high = pos_high[-1]
    print(f"p = {p_low}:  endpoint = {end_low},  |X_n| = {np.linalg.norm(end_low):.1f}")
    print(f"p = {p_high}: endpoint = {end_high}, |X_n| = {np.linalg.norm(end_high):.1f}")


if __name__ == "__main__":
    main()